In [1]:
import numpy as np
import pandas as pd

In [3]:
similarity_matrix = np.load("similarity_matrix.npy")

employee_performance = pd.read_csv("employee_performance_ready.csv")

employee_sentiment = pd.read_csv("predictions_with_sentiment.csv")

In [4]:
similarity_matrix.shape

(782, 782)

In [5]:
similarity_matrix.size

611524

In [6]:
print("Similarity Matrix Shape:", similarity_matrix.shape)
print("Employee Performance Shape:", employee_performance.shape)
print("Employee Sentiment Shape:", employee_sentiment.shape)

Similarity Matrix Shape: (782, 782)
Employee Performance Shape: (782, 4)
Employee Sentiment Shape: (782, 8)


In [7]:
num_employees, num_roles = similarity_matrix.shape

In [8]:
assert num_employees == len(employee_performance), "Employee count mismatch with performance data"
assert num_employees == len(employee_sentiment), "Employee count mismatch with sentiment data"

In [ ]:
#ALIGN DATA BY EMPLOYEE_ID

In [9]:
employee_performance = employee_performance.sort_values("employee_id").reset_index(drop=True)
employee_sentiment = employee_sentiment.sort_values("employee_id").reset_index(drop=True)

In [10]:
assert all(employee_performance["employee_id"] == employee_sentiment["employee_id"]), \
    "Employee IDs are not aligned between datasets"

print("Employee alignment check PASSED")

Employee alignment check PASSED


In [11]:
# extract the components

In [12]:
skill_cost_matrix = 1 - similarity_matrix

In [13]:
employee_perf_cost = employee_performance["employee_perf_cost"].values

In [14]:
sentiment_score = employee_sentiment["vader_score_0_1"].values
sentiment_cost = 1 - sentiment_score

In [15]:
employee_perf_cost = employee_perf_cost.reshape(-1, 1)
sentiment_cost = sentiment_cost.reshape(-1, 1)

In [26]:
#DEFINE WEIGHTS


In [27]:
skill_score = similarity_matrix.mean(axis=1)

X = np.vstack([
    skill_score,
    employee_perf_cost.flatten(),
    sentiment_cost.flatten()
]).T

print("Feature Matrix Shape:", X.shape)
print("First 5 rows:\n", X[:5])

Feature Matrix Shape: (782, 3)
First 5 rows:
 [[0.49003601 0.44456039 0.14555   ]
 [0.55131316 0.44456039 0.2798    ]
 [0.50720817 0.44456039 0.7608    ]
 [0.39631718 0.44456039 0.18155   ]
 [0.34487462 0.44456039 0.5       ]]


In [28]:
X_norm = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0) + 1e-9)

print("Normalized Matrix Shape:", X_norm.shape)
print("First 5 rows:\n", X_norm[:5])

Normalized Matrix Shape: (782, 3)
First 5 rows:
 [[0.63965519 0.46489993 0.14411554]
 [0.79118088 0.46489993 0.2816245 ]
 [0.68211837 0.46489993 0.77430093]
 [0.40790795 0.46489993 0.18098945]
 [0.28070115 0.46489993 0.50716993]]


In [29]:
p = X_norm / (X_norm.sum(axis=0) + 1e-9)

print("Probability Matrix Shape:", p.shape)
print("Column sums:", p.sum(axis=0))

Probability Matrix Shape: (782, 3)
Column sums: [1. 1. 1.]


In [30]:
entropy = -np.sum(p * np.log(p + 1e-9), axis=0) / np.log(len(X_norm))

print("Entropy values:", entropy)

Entropy values: [0.99007648 0.98581156 0.96502337]


In [31]:
divergence = 1 - entropy

print("Divergence values:", divergence)

Divergence values: [0.00992352 0.01418844 0.03497663]


In [32]:
entropy_weights = divergence / divergence.sum()

print("Entropy Weights:", entropy_weights)

Entropy Weights: [0.16794309 0.24012148 0.59193543]


In [33]:
preference = np.array([3,2,1])
preference_weights = preference / preference.sum()

print("Preference Weights:", preference_weights)

Preference Weights: [0.5        0.33333333 0.16666667]


In [41]:
alpha = 0.01

final_weights = alpha * entropy_weights + (1-alpha) * preference_weights

final_weights = final_weights / final_weights.sum()

print("Final Hybrid Weights:", final_weights)

Final Hybrid Weights: [0.49667943 0.33240121 0.17091935]


In [42]:
w_skill, w_perf, w_sent = final_weights

cost_matrix = (
    w_skill * skill_cost_matrix +
    w_perf * employee_perf_cost +
    w_sent * sentiment_cost
)

print("Cost Matrix Shape:", cost_matrix.shape)
print("Cost Matrix Stats:")
print("Min:", cost_matrix.min())
print("Max:", cost_matrix.max())
print("Mean:", cost_matrix.mean())

Cost Matrix Shape: (782, 782)
Cost Matrix Stats:
Min: 0.11369470745870487
Max: 0.8094053936583522
Mean: 0.4888274037074753


In [44]:
print("\nFinal Cost Matrix Shape:", cost_matrix.shape)

# Check NaN
print("NaN values:", np.isnan(cost_matrix).sum())

# Check infinite
print("Infinite values:", np.isinf(cost_matrix).sum())

# Range check
print("Cost Matrix Min:", cost_matrix.min())
print("Cost Matrix Max:", cost_matrix.max())


Final Cost Matrix Shape: (782, 782)
NaN values: 0
Infinite values: 0
Cost Matrix Min: 0.11369470745870487
Cost Matrix Max: 0.8094053936583522


In [45]:
print("\nSample Cost Matrix (first 5x5 block):")
print(cost_matrix[:5, :5])


Sample Cost Matrix (first 5x5 block):
[[0.24267068 0.34646634 0.33808685 0.3421391  0.40720205]
 [0.3449905  0.26968546 0.39823481 0.40466679 0.4302002 ]
 [0.50676249 0.5306872  0.40821862 0.5324632  0.55756448]
 [0.45546791 0.45668471 0.48959101 0.31406616 0.49564132]
 [0.54557842 0.53068782 0.48201484 0.52943306 0.36794778]]


In [46]:
from scipy.optimize import linear_sum_assignment

row_ind, col_ind = linear_sum_assignment(cost_matrix)

print("Assignments:", len(row_ind))
print("First 10 assignments:")
print(list(zip(row_ind[:10], col_ind[:10])))

Assignments: 782
First 10 assignments:
[(np.int64(0), np.int64(484)), (np.int64(1), np.int64(720)), (np.int64(2), np.int64(701)), (np.int64(3), np.int64(255)), (np.int64(4), np.int64(723)), (np.int64(5), np.int64(121)), (np.int64(6), np.int64(221)), (np.int64(7), np.int64(502)), (np.int64(8), np.int64(8)), (np.int64(9), np.int64(9))]


In [47]:
employee_ids = employee_performance.sort_values("employee_id")["employee_id"].values

In [48]:
print(employee_ids[:10])

[108201 108201 108201 108201 108201 108201 108205 108205 108205 108205]


In [50]:
team_df = pd.read_csv("team_dataset_realistic.csv")

In [52]:
roles = team_df[["team_id","job_role"]].values

In [53]:
print(roles[:5])

[['TEAM_001' 'Program Manager']
 ['TEAM_001' 'Project Manager']
 ['TEAM_001' 'Tech Lead']
 ['TEAM_001' 'Business Analyst']
 ['TEAM_001' 'UI/UX Designer']]


In [54]:
assignments = []

for emp_idx, role_idx in zip(row_ind, col_ind):

    employee_id = employee_ids[emp_idx]

    team_id = roles[role_idx][0]
    role = roles[role_idx][1]

    assignments.append([employee_id, team_id, role])

assignment_df = pd.DataFrame(assignments, columns=[
    "employee_id",
    "team_id",
    "assigned_role"
])

print(assignment_df.head(10))

   employee_id   team_id       assigned_role
0       108201  TEAM_063     Program Manager
1       108201  TEAM_094     Project Manager
2       108201  TEAM_091           Tech Lead
3       108201  TEAM_033    Business Analyst
4       108201  TEAM_094      UI/UX Designer
5       108201  TEAM_015  Software Developer
6       108205  TEAM_028  Software Developer
7       108205  TEAM_065  Software Developer
8       108205  TEAM_001    QA/Test Engineer
9       108205  TEAM_001    QA/Test Engineer


In [55]:
assignment_df.to_csv("optimized_employee_assignment.csv", index=False)